In [1]:
# ==========================================
# PHASE 0: IMPORT LIBRARIES
# ==========================================
import time
import numpy as np
import pandas as pd

from scipy.stats import randint
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# ==========================================
# PHASE 1: DATA ARCHITECTURE
# ==========================================
print("PHASE 1: DATA GENERATION")

# Generate synthetic placement dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    weights=[0.9, 0.1],   # 90% placed, 10% unplaced
    random_state=42
)

# Split into 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

# Feature Scaling (Golden Rule: fit only on training data)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed without data leakage ✅")

PHASE 1: DATA GENERATION
Training shape: (800, 20)
Testing shape: (200, 20)
Scaling completed without data leakage ✅


In [3]:
# ==========================================
# PHASE 2A: BASELINE MODEL
# ==========================================
print("\nPHASE 2A: BASELINE MODEL")

baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

baseline_pred = baseline_model.predict(X_test_scaled)

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)

print("Baseline Accuracy:", round(baseline_acc, 4))
print("Baseline F1 Score:", round(baseline_f1, 4))


PHASE 2A: BASELINE MODEL
Baseline Accuracy: 0.915
Baseline F1 Score: 0.2609


In [4]:
# ==========================================
# PHASE 2B: GRID SEARCH (ACCURACY)
# ==========================================
print("\nPHASE 2B: GRID SEARCH FOR ACCURACY")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_acc = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_acc.fit(X_train_scaled, y_train)

print("Best Parameters (Accuracy):")
print(grid_acc.best_params_)

acc_pred = grid_acc.predict(X_test_scaled)

print("Accuracy Optimized Model Accuracy:",
      round(accuracy_score(y_test, acc_pred), 4))
print("Accuracy Optimized Model F1:",
      round(f1_score(y_test, acc_pred), 4))


PHASE 2B: GRID SEARCH FOR ACCURACY
Best Parameters (Accuracy):
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Accuracy Optimized Model Accuracy: 0.915
Accuracy Optimized Model F1: 0.2609


In [5]:
# ==========================================
# PHASE 2C: GRID SEARCH (F1 SCORE)
# ==========================================
print("\nPHASE 2C: GRID SEARCH FOR F1 SCORE")

grid_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_f1.fit(X_train_scaled, y_train)

print("Best Parameters (F1):")
print(grid_f1.best_params_)

f1_pred = grid_f1.predict(X_test_scaled)

print("F1 Optimized Model Accuracy:",
      round(accuracy_score(y_test, f1_pred), 4))
print("F1 Optimized Model F1:",
      round(f1_score(y_test, f1_pred), 4))


PHASE 2C: GRID SEARCH FOR F1 SCORE
Best Parameters (F1):
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
F1 Optimized Model Accuracy: 0.915
F1 Optimized Model F1: 0.2609


In [6]:
# ==========================================
# PHASE 2D: METRIC TRAP ANALYSIS
# ==========================================
print("\nPHASE 2D: METRIC TRAP ANALYSIS")

print("""
Why does the best model change?

1. Accuracy focuses on overall correct predictions.
2. Since 90% students are placed, the model can predict mostly 'placed'
   and still get high accuracy.
3. F1-score balances precision + recall for minority class.
4. It focuses on correctly identifying 'at-risk' students.
5. Hence best hyperparameters change based on optimization metric.
""")


PHASE 2D: METRIC TRAP ANALYSIS

Why does the best model change?

1. Accuracy focuses on overall correct predictions.
2. Since 90% students are placed, the model can predict mostly 'placed'
   and still get high accuracy.
3. F1-score balances precision + recall for minority class.
4. It focuses on correctly identifying 'at-risk' students.
5. Hence best hyperparameters change based on optimization metric.



In [7]:
# ==========================================
# PHASE 3A: GRID SEARCH TIMING
# ==========================================
print("\nPHASE 3A: EFFICIENCY WARFARE - GRID SEARCH")

start_grid = time.time()

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

grid_time = time.time() - start_grid

grid_best_pred = grid_search.predict(X_test_scaled)
grid_best_f1 = f1_score(y_test, grid_best_pred)

print("Grid Search Time:", round(grid_time, 2), "seconds")
print("Best Grid F1:", round(grid_best_f1, 4))


PHASE 3A: EFFICIENCY WARFARE - GRID SEARCH
Grid Search Time: 9.45 seconds
Best Grid F1: 0.2609


In [8]:
# ==========================================
# PHASE 3B: RANDOMIZED SEARCH
# ==========================================
print("\nPHASE 3B: RANDOMIZED SEARCH")

param_dist = {
    'n_estimators': randint(10, 500),
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': randint(2, 20)
}

start_rand = time.time()

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_scaled, y_train)

rand_time = time.time() - start_rand

rand_best_pred = random_search.predict(X_test_scaled)
rand_best_f1 = f1_score(y_test, rand_best_pred)

print("Randomized Search Time:", round(rand_time, 2), "seconds")
print("Best Randomized F1:", round(rand_best_f1, 4))


PHASE 3B: RANDOMIZED SEARCH
Randomized Search Time: 20.23 seconds
Best Randomized F1: 0.2609


In [9]:
# ==========================================
# PHASE 3C: FINAL COMPARISON TABLE
# ==========================================
print("\nPHASE 3C: FINAL COMPARISON")

comparison = pd.DataFrame({
    "Method": ["Grid Search", "Randomized Search"],
    "Time Taken (sec)": [round(grid_time, 2), round(rand_time, 2)],
    "Best F1 Score": [round(grid_best_f1, 4), round(rand_best_f1, 4)]
})

comparison


PHASE 3C: FINAL COMPARISON


,Method,Time Taken (sec),Best F1 Score
0,Grid Search,9.45,0.2609
1,Randomized Search,20.23,0.2609


Conclusion:
- Accuracy is misleading for imbalanced datasets.
- F1-score is better for identifying at-risk students.
- Randomized Search gives near-best performance in less time.
- Better for large real-world datasets.